# 使用HuggingFace中的Trainer进行BERT模型微调

## 模型微调

In [ ]:
import datasets
from transformers import AutoTokenizer, DataCollatorForTokenClassification
from transformers import Trainer, TrainingArguments
from transformers import AutoModelForTokenClassification


model_id = 'bert-base-chinese'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(model_id, num_labels=7)
raw_datasets = datasets.load_dataset('peoples_daily_ner')
max_length = 128


def add_text(sample):
    text = []
    for item in sample['tokens']:
        text.append(''.join(item))
    return {"text": text}


processed_dataset = raw_datasets.map(add_text, batched=True)


def tokenize_function(sample):
    features = tokenizer(
        sample['text'], max_length=max_length, padding=True, truncation=True)
    features['labels'] = []
    for i in range(len(sample['ner_tags'])):
        features['labels'].append(
            [-100] + sample['ner_tags'][i][:max_length-2])
    return features


tokenized_datasets = processed_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(output_dir='ner_model',
                                  evaluation_strategy="epoch",
                                  per_device_train_batch_size=64,
                                  per_device_eval_batch_size=32,
                                  learning_rate=1e-4,
                                  num_train_epochs=10,
                                  warmup_ratio=0.2,
                                  logging_dir='ner_log',
                                  logging_strategy="epoch",
                                  save_strategy="epoch",
                                  report_to="tensorboard")

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

trainer.train()

## 模型测试

In [ ]:
import datasets
import numpy as np
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from seqeval.metrics import classification_report
from tqdm import tqdm

max_length = 128
checkpoint = "./ner_model/checkpoint-3270"
model = AutoModelForTokenClassification.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
label_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


def single_predict(text):
    encoding = tokenizer(text, max_length=max_length,
                         padding=True, truncation=True, return_tensors='pt')
    logits = model(**encoding).logits
    prediction = np.argmax(logits.detach().numpy(), axis=2)[0]
    return [label_list[_] for _ in prediction[1:-1]]


text = "王安石的另一位好友叫吴充。"
t = single_predict(text=text)
print(t)

raw_datasets = datasets.load_dataset('peoples_daily_ner')['test']
true_tag_list, pred_tag_list = [], []
for i in tqdm(range(len(raw_datasets))):
    test_text = ''.join(raw_datasets[i]['tokens'])
    true_tag = [label_list[_] for _ in raw_datasets[i]['ner_tags']]
    pred_tag = single_predict(text=test_text)
    true_tag_list.append(true_tag)
    if len(true_tag) <= len(pred_tag):
        pred_tag_list.append(pred_tag[:len(true_tag)])
    else:
        pred_tag_list.append(pred_tag+["O"]*(len(true_tag)-len(pred_tag)))

print(classification_report(true_tag_list, pred_tag_list, digits=4))